import mysql connector

In [1]:
import mysql.connector

connect to mysql server

In [2]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


create database

In [3]:
cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS ecommerce_db")

print("Database created successfully!")

Database created successfully!


connect to the new database

In [5]:
conn.close()

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecommerce_db"
)

cursor = conn.cursor()

print("Connected to ecommerce_db successfully!")

Connected to ecommerce_db successfully!


database verification

In [6]:
cursor.execute("SELECT DATABASE()")

database_name = cursor.fetchone()[0]

print("Current database:", database_name)

Current database: ecommerce_db


create customer table

In [7]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(50),
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix INT,
    customer_city VARCHAR(100),
    customer_state VARCHAR(10)
)
""")

print("CUSTOMERS table created")

CUSTOMERS table created


create order table

In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(50),
    customer_id VARCHAR(50),
    order_status VARCHAR(30),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME
)
""")

print("ORDERS table created")

ORDERS table created


create product table

In [9]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(50),
    product_category_name VARCHAR(150),
    product_name_length INT,
    product_description_length INT,
    product_photos_qty INT,
    product_weight_g FLOAT,
    product_length_cm FLOAT,
    product_height_cm FLOAT,
    product_width_cm FLOAT
)
""")

print("PRODUCTS table created")

PRODUCTS table created


create seller table

In [10]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(50),
    seller_zip_code_prefix INT,
    seller_city VARCHAR(100),
    seller_state VARCHAR(10)
)
""")

print("SELLERS table created")

SELLERS table created


create geolocation table

In [11]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix INT,
    geolocation_lat FLOAT,
    geolocation_lng FLOAT,
    geolocation_city VARCHAR(100),
    geolocation_state VARCHAR(10)
)
""")

print("GEOLOCATION table created")

GEOLOCATION table created


create product_category_translation table

In [12]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS product_category_translation (
    product_category_name VARCHAR(150),
    product_category_name_english VARCHAR(150)
)
""")

print("PRODUCT_CATEGORY_TRANSLATION table created")

PRODUCT_CATEGORY_TRANSLATION table created


create orders_payments table

In [13]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_payments (
    order_id VARCHAR(50),
    payment_sequential INT,
    payment_type VARCHAR(30),
    payment_installments INT,
    payment_value DECIMAL(12,2)
)
""")

print("ORDER_PAYMENTS table created")

ORDER_PAYMENTS table created


create order_reviews table

In [14]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_reviews (
    review_id VARCHAR(50),
    order_id VARCHAR(50),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME
)
""")

print("ORDER_REVIEWS table created")

ORDER_REVIEWS table created


create order_item table

In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(50),
    order_item_id INT,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),
    shipping_limit_date DATETIME,
    price DECIMAL(12,2),
    freight_value DECIMAL(12,2)
)
""")

print("ORDER_ITEMS table created")

ORDER_ITEMS table created


commit the table creation

In [16]:
conn.commit()

print("All tables committed successfully!")

All tables committed successfully!


verify all 9 tables

In [17]:
cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

for table in tables:
    print(table[0])

customers
geolocation
order_items
order_payments
order_reviews
orders
product_category_translation
products
sellers


det cleaned data path

In [18]:
CLEANED_DATA = r"C:\Users\Abcom\Desktop\Decoding E-Commerce Platform\data\cleaned"

print("Cleaned data path:")
print(CLEANED_DATA)

Cleaned data path:
C:\Users\Abcom\Desktop\Decoding E-Commerce Platform\data\cleaned


import libraries

In [19]:
import pandas as pd
import os

load customers

In [ ]:
customers_df = pd.read_csv(
    os.path.join(CLEANED_DATA, "customers_cleaned.csv")
)

print("Customers loaded:", len(customers_df))
display(customers_df.head())

insert customers into mysql

In [21]:
customers_query = """
INSERT INTO customers (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
)
VALUES (%s, %s, %s, %s, %s)
"""

customers_data = [
    tuple(row)
    for row in customers_df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ].itertuples(index=False, name=None)
]

cursor.executemany(customers_query, customers_data)

conn.commit()

print("Customers data loaded successfully.")

Customers data loaded successfully.


load orders

In [22]:
orders_df = pd.read_csv(
    os.path.join(CLEANED_DATA, "orders_cleaned.csv")
)

print("Orders loaded:", len(orders_df))
display(orders_df.head())

Orders loaded: 99441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [25]:
# Convert order date columns safely

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_df[column] = pd.to_datetime(
        orders_df[column],
        errors="coerce"
    )

# Convert pandas Timestamp / NaT
# into Python datetime / None
for column in date_columns:
    orders_df[column] = orders_df[column].apply(
        lambda x: x.to_pydatetime() if pd.notna(x) else None
    )

print("Order date columns prepared successfully.")

Order date columns prepared successfully.


In [28]:
# Convert pandas datetime columns to Python datetime objects
# and replace missing dates with None

for column in date_columns:
    orders_df[column] = orders_df[column].apply(
        lambda x: x.to_pydatetime() if pd.notna(x) else None
    )

print("Date conversion completed.")

print(type(orders_df["order_purchase_timestamp"].iloc[0]))
print(type(orders_df["order_approved_at"].iloc[0]))

Date conversion completed.
<class 'pandas.Timestamp'>
<class 'pandas.Timestamp'>


In [29]:
print(
    orders_df[date_columns].isna().sum()
)

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


recreate the date column

In [32]:
# Prepare order date columns for MySQL

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    # Convert to datetime
    orders_df[column] = pd.to_datetime(
        orders_df[column],
        errors="coerce"
    )

    # Convert valid dates to MySQL-compatible strings
    orders_df[column] = orders_df[column].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    # Convert missing values (NaN) to None
    orders_df[column] = orders_df[column].where(
        orders_df[column].notna(),
        None
    )

print("Order dates prepared for MySQL.")

Order dates prepared for MySQL.


In [33]:
print(orders_df[date_columns].head())

  order_purchase_timestamp    order_approved_at order_delivered_carrier_date  \
0      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00   
1      2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00   
2      2018-08-08 08:38:49  2018-08-08 08:55:23          2018-08-08 13:50:00   
3      2017-11-18 19:28:06  2017-11-18 19:45:59          2017-11-22 13:39:59   
4      2018-02-13 21:18:39  2018-02-13 22:20:29          2018-02-14 19:46:34   

  order_delivered_customer_date order_estimated_delivery_date  
0           2017-10-10 21:25:13           2017-10-18 00:00:00  
1           2018-08-07 15:27:45           2018-08-13 00:00:00  
2           2018-08-17 18:06:29           2018-09-04 00:00:00  
3           2017-12-02 00:28:42           2017-12-15 00:00:00  
4           2018-02-16 18:17:02           2018-02-26 00:00:00  


In [34]:
print(type(orders_df["order_purchase_timestamp"].iloc[0]))

<class 'str'>


In [35]:
print(
    orders_df["order_approved_at"].isna().sum()
)

160


inserting the orders

In [36]:
orders_query = """
INSERT INTO orders (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

orders_data = list(
    orders_df[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    orders_query,
    orders_data
)

conn.commit()

print("Orders data loaded successfully.")

Orders data loaded successfully.


In [37]:
cursor.execute("SELECT COUNT(*) FROM orders")

count = cursor.fetchone()[0]

print("Orders in MySQL:", count)

Orders in MySQL: 99441


load products

In [38]:
products_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "products_cleaned.csv"
    )
)

print("Products loaded:", len(products_df))

Products loaded: 32951


insert products into mysql

In [39]:
products_query = """
INSERT INTO products (
    product_id,
    product_category_name,
    product_name_length,
    product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

products_data = list(
    products_df[
        [
            "product_id",
            "product_category_name",
            "product_name_length",
            "product_description_length",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    products_query,
    products_data
)

conn.commit()

print("Products data loaded successfully.")

Products data loaded successfully.


load sellers dataset

In [41]:
sellers_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "sellers_cleaned.csv"
    )
)

print("Sellers loaded:", len(sellers_df))

Sellers loaded: 3095


prepare missing values

In [42]:
sellers_df = sellers_df.where(
    pd.notnull(sellers_df),
    None
)

print("Sellers data prepared.")

Sellers data prepared.


insert sellers into mysql

In [43]:
sellers_query = """
INSERT INTO sellers (
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
VALUES (%s, %s, %s, %s)
"""

sellers_data = list(
    sellers_df[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    sellers_query,
    sellers_data
)

conn.commit()

print("Sellers data loaded successfully.")

Sellers data loaded successfully.


load catagory translation

In [45]:
category_translation_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "category_translation_cleaned.csv"
    )
)

print("Category translation loaded:", len(category_translation_df))

Category translation loaded: 71


prepare the data

In [46]:
category_translation_df = category_translation_df.where(
    pd.notnull(category_translation_df),
    None
)

print("Category translation data prepared.")

Category translation data prepared.


insert into mysql

In [49]:
category_query = """
INSERT INTO product_category_translation (
    product_category_name,
    product_category_name_english
)
VALUES (%s, %s)
"""

category_data = list(
    category_translation_df[
        [
            "product_category_name",
            "product_category_name_english"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    category_query,
    category_data
)

conn.commit()

print("Product category translation data loaded successfully.")

Product category translation data loaded successfully.


load order review datasets

In [51]:
order_reviews_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "reviews_cleaned.csv"
    )
)

print("Reviews loaded:", len(order_reviews_df))

Reviews loaded: 99224


prepare missing values

In [52]:
order_reviews_df = order_reviews_df.where(
    pd.notnull(order_reviews_df),
    None
)

print("Reviews data prepared.")

Reviews data prepared.


prepare the review data column

In [54]:
# Prepare review date columns for MySQL

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    order_reviews_df[column] = pd.to_datetime(
        order_reviews_df[column],
        errors="coerce"
    )

    order_reviews_df[column] = order_reviews_df[column].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    order_reviews_df[column] = order_reviews_df[column].where(
        order_reviews_df[column].notna(),
        None
    )

print("Review dates prepared for MySQL.")

Review dates prepared for MySQL.


insert order reviews into mysql

In [55]:
reviews_query = """
INSERT INTO order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

reviews_data = list(
    order_reviews_df[
        [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    reviews_query,
    reviews_data
)

conn.commit()

print("Reviews data loaded successfully.")

Reviews data loaded successfully.


load order payment dataset

In [57]:
order_payments_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "payments_cleaned.csv"
    )
)

print("Payments loaded:", len(order_payments_df))

Payments loaded: 103884


prepare missing values

In [58]:
order_payments_df = order_payments_df.where(
    pd.notnull(order_payments_df),
    None
)

print("Payments data prepared.")

Payments data prepared.


In [59]:
print(order_payments_df.columns.tolist())

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


insert order payments into mysql

In [60]:
payments_query = """
INSERT INTO order_payments (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
VALUES (%s, %s, %s, %s, %s)
"""

payments_data = list(
    order_payments_df[
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    payments_query,
    payments_data
)

conn.commit()

print("Payments data loaded successfully.")

Payments data loaded successfully.


load the geolocation dataset

In [62]:
geolocation_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "geolocation_cleaned.csv"
    )
)

print("Geolocation loaded:", len(geolocation_df))

Geolocation loaded: 738332


prepare missing values

In [63]:
geolocation_df = geolocation_df.where(
    pd.notnull(geolocation_df),
    None
)

print("Geolocation data prepared.")

Geolocation data prepared.


In [64]:
print(geolocation_df.columns.tolist())

['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


insert geolocation data into mysql

In [65]:
geolocation_query = """
INSERT INTO geolocation (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
)
VALUES (%s, %s, %s, %s, %s)
"""

geolocation_data = list(
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    geolocation_query,
    geolocation_data
)

conn.commit()

print("Geolocation data loaded successfully.")

Geolocation data loaded successfully.


load order item dataset

In [67]:
order_items_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "order_items_cleaned.csv"
    )
)

print("Order items loaded:", len(order_items_df))

Order items loaded: 112650


prepare missing value

In [68]:
order_items_df = order_items_df.where(
    pd.notnull(order_items_df),
    None
)

print("Order items data prepared.")

Order items data prepared.


In [69]:
print(order_items_df.columns.tolist())


['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


prepare the shipping date

In [70]:
# Prepare shipping date for MySQL

order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"],
    errors="coerce"
)

order_items_df["shipping_limit_date"] = (
    order_items_df["shipping_limit_date"]
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

order_items_df["shipping_limit_date"] = (
    order_items_df["shipping_limit_date"].where(
        order_items_df["shipping_limit_date"].notna(),
        None
    )
)

print("Shipping date prepared for MySQL.")

Shipping date prepared for MySQL.


insert order items into mysql

In [71]:
order_items_query = """
INSERT INTO order_items (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

order_items_data = list(
    order_items_df[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    order_items_query,
    order_items_data
)

conn.commit()

print("Order items data loaded successfully.")

Order items data loaded successfully.


load customer dataset

In [73]:
customers_df = pd.read_csv(
    os.path.join(
        CLEANED_DATA,
        "customers_cleaned.csv"
    )
)

print("Customers loaded:", len(customers_df))

Customers loaded: 99441


prepare missing values


In [74]:
customers_df = customers_df.where(
    pd.notnull(customers_df),
    None
)

print("Customers data prepared.")

Customers data prepared.


In [75]:
print(customers_df.columns.tolist())

['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [78]:
cursor.execute("DELETE FROM customers")
conn.commit()

print("Customers table cleared.")

Customers table cleared.


In [79]:
cursor.execute("SELECT COUNT(*) FROM customers")

print("Customers remaining:", cursor.fetchone()[0])

Customers remaining: 0


insert customer value into mysql

In [80]:
customers_query = """
INSERT INTO customers (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
)
VALUES (%s, %s, %s, %s, %s)
"""

customers_data = list(
    customers_df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(
    customers_query,
    customers_data
)

conn.commit()

print("Customers data loaded successfully.")

Customers data loaded successfully.


In [81]:
cursor.execute("SELECT COUNT(*) FROM customers")

customers_count = cursor.fetchone()[0]

print("Customers in MySQL:", customers_count)

Customers in MySQL: 99441


add primary key

In [83]:
cursor.execute("""
SELECT
    TABLE_NAME,
    CONSTRAINT_NAME,
    COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'ecommerce_db'
AND CONSTRAINT_NAME = 'PRIMARY'
ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

primary_keys = cursor.fetchall()

for row in primary_keys:
    print(row)

('customers', 'PRIMARY', 'customer_id')
('orders', 'PRIMARY', 'order_id')
('product_category_translation', 'PRIMARY', 'product_category_name')
('products', 'PRIMARY', 'product_id')
('sellers', 'PRIMARY', 'seller_id')


In [84]:
cursor.execute("""
SELECT review_id, order_id, COUNT(*) AS cnt
FROM order_reviews
GROUP BY review_id, order_id
HAVING COUNT(*) > 1
LIMIT 10
""")

duplicate_review_keys = cursor.fetchall()

print(duplicate_review_keys)

[]


In [85]:
cursor.execute("""
ALTER TABLE order_reviews
ADD PRIMARY KEY (review_id, order_id)
""")

conn.commit()

print("order_reviews primary key added successfully.")

order_reviews primary key added successfully.


add the remaining primary key

In [86]:
primary_key_queries = [
    """
    ALTER TABLE order_payments
    ADD PRIMARY KEY (order_id, payment_sequential)
    """,

    """
    ALTER TABLE order_items
    ADD PRIMARY KEY (order_id, order_item_id)
    """
]

for query in primary_key_queries:
    cursor.execute(query)

conn.commit()

print("Remaining primary keys added successfully.")

Remaining primary keys added successfully.


verify everything

In [87]:
cursor.execute("""
SELECT
    TABLE_NAME,
    CONSTRAINT_NAME,
    COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'ecommerce_db'
AND CONSTRAINT_NAME = 'PRIMARY'
ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

primary_keys = cursor.fetchall()

for row in primary_keys:
    print(row)

('customers', 'PRIMARY', 'customer_id')
('order_items', 'PRIMARY', 'order_id')
('order_items', 'PRIMARY', 'order_item_id')
('order_payments', 'PRIMARY', 'order_id')
('order_payments', 'PRIMARY', 'payment_sequential')
('order_reviews', 'PRIMARY', 'review_id')
('order_reviews', 'PRIMARY', 'order_id')
('orders', 'PRIMARY', 'order_id')
('product_category_translation', 'PRIMARY', 'product_category_name')
('products', 'PRIMARY', 'product_id')
('sellers', 'PRIMARY', 'seller_id')


foreign key orders to customers

In [88]:
cursor.execute("""
ALTER TABLE orders
ADD CONSTRAINT fk_orders_customer
FOREIGN KEY (customer_id)
REFERENCES customers(customer_id)
""")

conn.commit()

print("Orders → Customers foreign key added successfully.")

Orders → Customers foreign key added successfully.


order_items to orders

In [89]:
cursor.execute("""
ALTER TABLE order_items
ADD CONSTRAINT fk_order_items_order
FOREIGN KEY (order_id)
REFERENCES orders(order_id)
""")

conn.commit()

print("Order Items → Orders foreign key added successfully.")

Order Items → Orders foreign key added successfully.


order_items to products

In [90]:
cursor.execute("""
ALTER TABLE order_items
ADD CONSTRAINT fk_order_items_product
FOREIGN KEY (product_id)
REFERENCES products(product_id)
""")

conn.commit()

print("Order Items → Products foreign key added successfully.")

Order Items → Products foreign key added successfully.


order_items to sellers

In [91]:
cursor.execute("""
ALTER TABLE order_items
ADD CONSTRAINT fk_order_items_seller
FOREIGN KEY (seller_id)
REFERENCES sellers(seller_id)
""")

conn.commit()

print("Order Items → Sellers foreign key added successfully.")

Order Items → Sellers foreign key added successfully.


order_payments to orders

In [92]:
cursor.execute("""
ALTER TABLE order_payments
ADD CONSTRAINT fk_order_payments_order
FOREIGN KEY (order_id)
REFERENCES orders(order_id)
""")

conn.commit()

print("Order Payments → Orders foreign key added successfully.")


Order Payments → Orders foreign key added successfully.


order_reviews to orders

In [93]:
cursor.execute("""
ALTER TABLE order_reviews
ADD CONSTRAINT fk_order_reviews_order
FOREIGN KEY (order_id)
REFERENCES orders(order_id)
""")

conn.commit()

print("Order Reviews → Orders foreign key added successfully.")

Order Reviews → Orders foreign key added successfully.


In [94]:
cursor.execute("""
SELECT COUNT(*)
FROM products p
LEFT JOIN product_category_translation pct
    ON p.product_category_name = pct.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND pct.product_category_name IS NULL
""")

missing_categories = cursor.fetchone()[0]

print("Products with missing category references:", missing_categories)

Products with missing category references: 623


show the missing category

In [95]:
cursor.execute("""
SELECT
    p.product_category_name,
    COUNT(*) AS product_count
FROM products p
LEFT JOIN product_category_translation pct
    ON p.product_category_name = pct.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND pct.product_category_name IS NULL
GROUP BY p.product_category_name
ORDER BY product_count DESC
""")

missing_categories = cursor.fetchall()

for row in missing_categories:
    print(row)

('unknown', 610)
('portateis_cozinha_e_preparadores_de_alimentos', 10)
('pc_gamer', 3)


In [96]:
cursor.execute("""
SELECT
    product_category_name,
    product_category_name_english
FROM product_category_translation
ORDER BY product_category_name
""")

categories = cursor.fetchall()

for row in categories:
    print(row)

('agro_industria_e_comercio', 'agro_industry_and_commerce')
('alimentos', 'food')
('alimentos_bebidas', 'food_drink')
('artes', 'art')
('artes_e_artesanato', 'arts_and_craftmanship')
('artigos_de_festas', 'party_supplies')
('artigos_de_natal', 'christmas_supplies')
('audio', 'audio')
('automotivo', 'auto')
('bebes', 'baby')
('bebidas', 'drinks')
('beleza_saude', 'health_beauty')
('brinquedos', 'toys')
('cama_mesa_banho', 'bed_bath_table')
('casa_conforto', 'home_confort')
('casa_conforto_2', 'home_comfort_2')
('casa_construcao', 'home_construction')
('cds_dvds_musicais', 'cds_dvds_musicals')
('cine_foto', 'cine_photo')
('climatizacao', 'air_conditioning')
('consoles_games', 'consoles_games')
('construcao_ferramentas_construcao', 'construction_tools_construction')
('construcao_ferramentas_ferramentas', 'costruction_tools_tools')
('construcao_ferramentas_iluminacao', 'construction_tools_lights')
('construcao_ferramentas_jardim', 'costruction_tools_garden')
('construcao_ferramentas_segura

In [97]:
cursor.execute("""
SELECT *
FROM product_category_translation
WHERE product_category_name IN (
    'unknown',
    'portateis_cozinha_e_preparadores_de_alimentos',
    'pc_gamer'
)
""")

print(cursor.fetchall())

[]


add the missing category

In [98]:
missing_category_data = [
    (
        'portateis_cozinha_e_preparadores_de_alimentos',
        'kitchen_small_appliances'
    ),
    (
        'pc_gamer',
        'gaming_pc'
    )
]

category_insert_query = """
INSERT INTO product_category_translation
(product_category_name, product_category_name_english)
VALUES (%s, %s)
"""

cursor.executemany(
    category_insert_query,
    missing_category_data
)

conn.commit()

print("Missing valid categories added successfully.")

Missing valid categories added successfully.


handle unknown

In [99]:
cursor.execute("""
SELECT COUNT(*)
FROM products
WHERE product_category_name = 'unknown'
""")

print("Unknown category products:", cursor.fetchone()[0])

Unknown category products: 610


check the remaing missing reference

In [100]:
cursor.execute("""
SELECT
    p.product_category_name,
    COUNT(*) AS product_count
FROM products p
LEFT JOIN product_category_translation pct
    ON p.product_category_name = pct.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND pct.product_category_name IS NULL
GROUP BY p.product_category_name
ORDER BY product_count DESC
""")

remaining_missing = cursor.fetchall()

for row in remaining_missing:
    print(row)

('unknown', 610)


add unknown

In [101]:
cursor.execute("""
INSERT INTO product_category_translation
(product_category_name, product_category_name_english)
VALUES ('unknown', 'unknown')
""")

conn.commit()

print("Unknown category added successfully.")

Unknown category added successfully.


verify the relationship

In [102]:
cursor.execute("""
SELECT COUNT(*)
FROM products p
LEFT JOIN product_category_translation pct
    ON p.product_category_name = pct.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND pct.product_category_name IS NULL
""")

missing_categories = cursor.fetchone()[0]

print("Products with missing category references:", missing_categories)

Products with missing category references: 0


add the foreign key

In [103]:
cursor.execute("""
ALTER TABLE products
ADD CONSTRAINT fk_products_category
FOREIGN KEY (product_category_name)
REFERENCES product_category_translation(product_category_name)
""")

conn.commit()

print("Products → Category foreign key added successfully.")

Products → Category foreign key added successfully.


rder_items to product

In [106]:
cursor.execute("""
SELECT
    CONSTRAINT_NAME,
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'ecommerce_db'
  AND TABLE_NAME = 'order_items'
  AND CONSTRAINT_NAME = 'fk_order_items_product'
""")

result = cursor.fetchall()

for row in result:
    print(row)

('fk_order_items_product', 'order_items', 'product_id', 'products', 'product_id')


order_items to sellers

In [109]:
cursor.execute("""
SELECT
    CONSTRAINT_NAME,
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'ecommerce_db'
  AND TABLE_NAME = 'order_items'
  AND CONSTRAINT_NAME = 'fk_order_items_seller'
""")

result = cursor.fetchall()

for row in result:
    print(row)
    

('fk_order_items_seller', 'order_items', 'seller_id', 'sellers', 'seller_id')


check all foreign keys

In [113]:
cursor.execute("""
SELECT
    CONSTRAINT_NAME,
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'ecommerce_db'
  AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME, CONSTRAINT_NAME;
""")

foreign_keys = cursor.fetchall()

for row in foreign_keys:
    print(row)

('fk_order_items_order', 'order_items', 'order_id', 'orders', 'order_id')
('fk_order_items_product', 'order_items', 'product_id', 'products', 'product_id')
('fk_order_items_seller', 'order_items', 'seller_id', 'sellers', 'seller_id')
('fk_order_payments_order', 'order_payments', 'order_id', 'orders', 'order_id')
('fk_order_reviews_order', 'order_reviews', 'order_id', 'orders', 'order_id')
('fk_orders_customer', 'orders', 'customer_id', 'customers', 'customer_id')
('fk_products_category', 'products', 'product_category_name', 'product_category_translation', 'product_category_name')


verify row count

In [114]:
tables = [
    "customers",
    "orders",
    "products",
    "sellers",
    "product_category_translation",
    "order_items",
    "order_payments",
    "order_reviews",
    "geolocation"
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`")
    count = cursor.fetchone()[0]
    print(f"{table}: {count}")

customers: 99441
orders: 99441
products: 32951
sellers: 3095
product_category_translation: 74
order_items: 112650
order_payments: 103884
order_reviews: 99224
geolocation: 738332


verify primary keys

In [115]:
cursor.execute("""
SELECT
    TABLE_NAME,
    CONSTRAINT_NAME,
    COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'ecommerce_db'
  AND CONSTRAINT_NAME = 'PRIMARY'
ORDER BY TABLE_NAME;
""")

for row in cursor.fetchall():
    print(row)

('customers', 'PRIMARY', 'customer_id')
('order_items', 'PRIMARY', 'order_id')
('order_items', 'PRIMARY', 'order_item_id')
('order_payments', 'PRIMARY', 'order_id')
('order_payments', 'PRIMARY', 'payment_sequential')
('order_reviews', 'PRIMARY', 'review_id')
('order_reviews', 'PRIMARY', 'order_id')
('orders', 'PRIMARY', 'order_id')
('product_category_translation', 'PRIMARY', 'product_category_name')
('products', 'PRIMARY', 'product_id')
('sellers', 'PRIMARY', 'seller_id')


check orphan record

In [116]:
checks = {
    "Orders with missing customers": """
        SELECT COUNT(*)
        FROM orders o
        LEFT JOIN customers c
        ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,

    "Order items with missing orders": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN orders o
        ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "Order items with missing products": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN products p
        ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL
    """,

    "Order items with missing sellers": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN sellers s
        ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL
    """,

    "Payments with missing orders": """
        SELECT COUNT(*)
        FROM order_payments op
        LEFT JOIN orders o
        ON op.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "Reviews with missing orders": """
        SELECT COUNT(*)
        FROM order_reviews r
        LEFT JOIN orders o
        ON r.order_id = o.order_id
        WHERE o.order_id IS NULL
    """
}

for name, query in checks.items():
    cursor.execute(query)
    print(f"{name}: {cursor.fetchone()[0]}")

Orders with missing customers: 0
Order items with missing orders: 0
Order items with missing products: 0
Order items with missing sellers: 0
Payments with missing orders: 0
Reviews with missing orders: 0
